In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
#for plots inside the notebook
%matplotlib inline

In [ ]:
file_path = str(PROJECT_ROOT / "Dataset Specs" / "hospital_occupancy_daily.csv")

df = pd.read_csv(file_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

daily_occupancy = df["occupied_beds"].asfreq("D")

print(daily_occupancy.head())
print(daily_occupancy.tail())

In [ ]:
test_days = 270

train = daily_occupancy.iloc[:-test_days]
test = daily_occupancy.iloc[-test_days:]

print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Training period:", train.index.min(), "to", train.index.max())
print("Testing period:", test.index.min(), "to", test.index.max())

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(train, order=(1, 0, 1))
model_fit = model.fit()

print(model_fit.summary())

In [ ]:
forecast = model_fit.get_forecast(steps=len(test))

y_pred = np.asarray(forecast.predicted_mean)
y_true = test.to_numpy()

print("Number of predictions:", len(y_pred))
print("Number of actual values:", len(y_true))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("ARIMA Performance — 270 Days")
print(f"RMSE: {rmse:.3f} beds")
print(f"MAE:  {mae:.3f} beds")
print(f"MAPE: {mape:.3f}%")

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(test.index, y_true, label="Actual Occupancy", color="black")
plt.plot(test.index, y_pred, label="ARIMA Forecast")

plt.title("ARIMA: Actual vs Predicted Occupancy — Final 270 Days")
plt.xlabel("Date")
plt.ylabel("Occupied Beds")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/270_Days"
)
output_dir.mkdir(parents=True, exist_ok=True)

arima_pred_df = pd.DataFrame({
    "date": test.index,
    "actual_occupied_beds": y_true,
    "arima_predicted_occupied_beds": y_pred
})

output_file = output_dir / "arima_occupancy_predictions.csv"
arima_pred_df.to_csv(output_file, index=False)

print("ARIMA predictions saved to:", output_file)
print("Rows saved:", len(arima_pred_df))